In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
pdfs = list(Path("./raw/").glob("*.pdf"))

In [3]:
import ollama
import fitz

In [ ]:
OLLAMA_URL = "http://localhost:11434"
TEXT_EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2-vision"

In [4]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
import concurrent
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
embeddings = OllamaEmbeddings(
    model="qwen3-embedding",
    base_url="http://localhost:11434"
)

In [ ]:
text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
)

In [5]:
from tqdm.auto import tqdm

In [ ]:
def semantic_chunking():
    embeddings = OllamaEmbeddings(
    model="qwen3-embedding",
    base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
    )
    docs = []
    for doc in pdfs:
        loader = PyPDFLoader(doc)

        docs.extend(loader.load())
        
    semantic_chunks = []
    for doc1 in tqdm(docs, desc="Splitting documents"):
        semantic_chunks.extend(text_splitter.split_documents([doc1]))

    return semantic_chunks

semantic_chunking()

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
import concurrent.futures
from tqdm import tqdm
# Ensure your Langchain/Ollama imports are here too

def load_single_pdf(pdf_path):
    """Helper function to load a single PDF."""
    loader = PyPDFLoader(pdf_path)
    return loader.load()

def chunk_single_doc(doc, text_splitter):
    """Helper function to chunk a single document."""
    return text_splitter.split_documents([doc])

def semantic_chunking(pdfs):
    embeddings = OllamaEmbeddings(
        model="mxbai-embed-large",
        base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
        embeddings, 
        breakpoint_threshold_type="percentile" 
    )
    pre_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,   # characters, not tokens
        chunk_overlap=200
    )
    docs = []
    # 1. Parallelize PDF Loading
    # ThreadPool is great here because file I/O releases the GIL.
    print("Starting parallel PDF loading...")
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Map applies the function to the iterable concurrently
        results = list(tqdm(executor.map(load_single_pdf, pdfs), total=len(pdfs), desc="Loading PDFs"))
        for res in results:
            docs.extend(res)
    
    docs = pre_splitter.split_documents(docs)
        
    semantic_chunks = []
    
    # 2. Parallelize Semantic Chunking
    # The chunker makes HTTP requests to Ollama, so threads work well.
    # CAUTION: Set max_workers carefully (see note below).
    MAX_WORKERS = 16
    
    print(f"Starting parallel chunking with {MAX_WORKERS} workers...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit tasks to the executor
        futures = [executor.submit(chunk_single_doc, doc, text_splitter) for doc in docs]
        
        # as_completed yields futures as they finish, keeping the progress bar accurate
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(docs), desc="Splitting documents"):
            try:
                semantic_chunks.extend(future.result())
            except Exception as e:
                print(f"Skipping a chunk due to error: {e}")  # Don't crash on one bad doc

    final_chunks = []

    for chunk in semantic_chunks:
        if len(chunk.page_content) > 1800:
            final_chunks.extend(pre_splitter.split_documents([chunk]))
        else:
            final_chunks.append(chunk)
    
    return final_chunks

semantic_chunks_parallel = semantic_chunking(pdfs)

Starting parallel PDF loading...


Loading PDFs: 100%|██████████| 18/18 [00:11<00:00,  1.62it/s]


Starting parallel chunking with 16 workers...


Splitting documents:  86%|████████▌ | 563/656 [19:55<03:42,  2.39s/it]

Skipping a chunk due to error: the input length exceeds the context length (status code: 400)


Splitting documents:  92%|█████████▏| 605/656 [21:25<01:36,  1.89s/it]

Skipping a chunk due to error: the input length exceeds the context length (status code: 400)
Skipping a chunk due to error: the input length exceeds the context length (status code: 400)


Splitting documents: 100%|██████████| 656/656 [22:57<00:00,  2.10s/it]


In [8]:
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction


In [13]:
client = chromadb.PersistentClient(path="chroma_db")

ef = OllamaEmbeddingFunction(
    model_name = "mxbai-embed-large",
    url="http://localhost:11434/api/embeddings"
)

In [14]:
collection = client.get_or_create_collection(name="semantic_texts", embedding_function=ef)

In [15]:
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ollama import ResponseError

embeddings_model = OllamaEmbeddings(model="mxbai-embed-large", base_url="http://localhost:11434")

# Re-split any oversized chunks (more aggressive threshold)
safety_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=100)
final_chunks = []
for chunk in semantic_chunks_parallel:
    if len(chunk.page_content) > 1200:
        final_chunks.extend(safety_splitter.split_documents([chunk]))
    else:
        final_chunks.append(chunk)

print(f"Before: {len(semantic_chunks_parallel)}, After: {len(final_chunks)}")

texts = [doc.page_content for doc in final_chunks]
metadatas = [doc.metadata for doc in final_chunks]

# Embed one-by-one to avoid any batch issues
all_embeddings = []
for i in tqdm(range(len(texts)), desc="Embedding"):
    try:
        emb = embeddings_model.embed_documents([texts[i]])
        all_embeddings.extend(emb)
    except ResponseError:
        print(f"Chunk {i} too long ({len(texts[i])} chars), sub-splitting...")
        sub = safety_splitter.split_text(texts[i])
        # Just embed the first sub-chunk to keep 1:1 mapping
        all_embeddings.extend(embeddings_model.embed_documents([sub[0]]))

assert len(all_embeddings) == len(texts), "Mismatch!"

collection.add(
    ids=[f"chunk_{i}" for i in range(len(texts))],
    documents=texts,
    metadatas=metadatas,
    embeddings=all_embeddings
)


Before: 1481, After: 1693


Embedding:   0%|          | 0/1693 [00:00<?, ?it/s]

Embedding: 100%|██████████| 1693/1693 [08:04<00:00,  3.50it/s]
